In [0]:
# create a spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Trading_sentiment_platform").getOrCreate()

In [0]:
# load risk text data into a dataframe
risk_df = spark.read.table("workspace.sec_filings.stg_cleaned_filings")\
            .select("cik", "company_name", "tickers", "filing_date", "accessionNumber", "form_type", "market_cap", "sector", "risks_text")
display(risk_df)

In [0]:
# Install spacy
# %pip install spacy==3.7.2
# !python -m spacy download en_core_web_sm

In [0]:
# %pip install --upgrade pip

In [0]:
# %restart_python

In [0]:
# Load the spacy model for risk analysis
import spacy
nlp = spacy.load("en_core_web_sm")

In [0]:
# define risk keywords
risk_keywords = [
    "risk", "inflation", "liability", "default", "fraud",
    "recession", "unemployment", "interest rates", "currency fluctuations",
    "tariffs", "trade restrictions", "geopolitical tensions", "conflict", "terrorism",
    "natural disasters", "earthquakes", "climate change", "extreme weather",
    "public health", "pandemic",
    "supply chain", "component shortages", "manufacturing disruption", "logistics",
    "cybersecurity", "ransomware", "data breach", "unauthorized access",
    "competition", "margin pressure", "obsolescence", "innovation",
    "regulation", "litigation", "intellectual property", "compliance",
    "credit risk", "liquidity",
    "labor disputes", "talent retention",
    "product defects", "recall", "quality issues"
]

In [0]:
# define risk term extraction function
def extract_risk_terms(text):
    if text is None or text.strip() == "":
        return {}

    doc = nlp(text)
    freq = {}

    for token in doc:
        lemma = token.lemma_.lower()
        if lemma in risk_keywords:
            freq[lemma] = freq.get(lemma, 0) + 1

    return freq


In [0]:
risk_pd = risk_df.toPandas()

# convert to pandas and run extraction
results = []
for _, row in risk_pd.iterrows():
    freq = extract_risk_terms(row["risks_text"])

    # convert dict to sorted list of top terms
    sorted_terms = sorted(freq.items(), key=lambda x: x[1], reverse=True)
    top_terms = [t[0] for t in sorted_terms[:3]] if sorted_terms else []

    results.append({
        "cik": row["cik"],
        "filing_date": row["filing_date"],
        "company_name": row["company_name"],
        "accession_number": row["accessionNumber"],
        "form_type": row["form_type"],
        "risk_term_count": sum(freq.values()),
        "top_terms": top_terms,
        "market_cap": row["market_cap"],
        "sector": row["sector"],
    })


In [0]:
# convert results back to spark dataframe
risk_df = spark.createDataFrame(results)
display(risk_df.limit(10))

In [0]:
target_schema = spark.table("sec_filings.fact_risk").schema
display(target_schema)

In [0]:
spark.sql("""CREATE OR REPLACE TABLE sec_filings.fact_risk (
    accession_number STRING,    
    cik STRING,
    company_name STRING,
    filing_date DATE,
    risk_term_count INTEGER,
    top_terms ARRAY<STRING>,
    processed_timestamp TIMESTAMP
)
USING DELTA;""")

# add timestamp to the risk dataframe
from pyspark.sql import functions as F

risk_df = risk_df.withColumn("processed_timestamp", F.current_timestamp())
display(risk_df)


In [0]:
from pyspark.sql.functions import col

# Align risk_df schema to match the Delta table
risk_df_aligned = risk_df.select(
    col("accession_number").cast("string"),
    col("cik").cast("string"),
    col("company_name").cast("string"),
    col("filing_date").cast("date"),
    col("risk_term_count").cast("int"),
    col("top_terms"),
    col("processed_timestamp").cast("timestamp")
)

In [0]:
# write the results to the fact_risk delta table
risk_df = risk_df.withColumn("risk_term_count", risk_df["risk_term_count"].cast('int'))
risk_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("sec_filings.fact_risk")


In [0]:
# %sql
# DROP TABLE IF EXISTS fact_risk